In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

import mlflow
import mlflow.sklearn

import dagshub
dagshub.init(repo_owner='dkhak22', repo_name='ml-assignment-2', mlflow=True)
mlflow.set_experiment('LogisticRegression_Training')

RANDOM_STATE = 42

In [ ]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity     = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

train = train_transaction.merge(train_identity, on='TransactionID', how='left')
test  = test_transaction.merge(test_identity,  on='TransactionID', how='left')

del train_transaction, train_identity, test_transaction, test_identity

print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Fraud rate: {train["isFraud"].mean():.4f}')

# Cleaning

In [ ]:
missing_pct = train.isnull().mean().sort_values(ascending=False)
print('Top 20 columns by missing %:')
print(missing_pct.head(20))

In [ ]:
MISSING_THRESHOLD = 0.5
drop_high_missing = missing_pct[missing_pct > MISSING_THRESHOLD].index.tolist()
drop_id = ['TransactionID']
DROP_COLS = list(set(drop_high_missing + drop_id))

train_clean = train.drop(columns=DROP_COLS)
test_clean  = test.drop(columns=[c for c in DROP_COLS if c in test.columns])

print(f'Columns before: {train.shape[1]}')
print(f'Dropped (>{MISSING_THRESHOLD*100:.0f}% missing): {len(drop_high_missing)}')
print(f'Columns after:  {train_clean.shape[1]}')

In [ ]:
with mlflow.start_run(run_name='LR_Cleaning'):
    mlflow.log_param('missing_threshold', MISSING_THRESHOLD)
    mlflow.log_param('n_dropped_high_missing', len(drop_high_missing))
    mlflow.log_metric('fraud_rate', train_clean['isFraud'].mean())
    mlflow.log_metric('n_cols_after', train_clean.shape[1])

    dropped_df = pd.DataFrame({
        'column': drop_high_missing,
        'missing_pct': [missing_pct[c] for c in drop_high_missing]
    }).sort_values('missing_pct', ascending=False)
    dropped_df.to_csv('/tmp/lr_dropped_columns.csv', index=False)
    mlflow.log_artifact('/tmp/lr_dropped_columns.csv')
    print('LR_Cleaning run logged.')

# Feature Engineering

In [ ]:
# Columns with high cardinality — frequency-encode these instead of OHE
HIGH_CARD_COLS = [
    'card1', 'card2', 'card3', 'card5',
    'P_emaildomain', 'R_emaildomain',
    'DeviceInfo', 'id_31', 'id_33', 'id_20'
]
HIGH_CARD_COLS = [c for c in HIGH_CARD_COLS if c in train_clean.columns]

In [ ]:
class FraudFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Custom transformer that adds time features, log-transforms TransactionAmt,
    and applies frequency encoding for high-cardinality categoricals.
    All statistics are learned from training data only.
    """
    def __init__(self, freq_cols=None):
        self.freq_cols = freq_cols or []

    def fit(self, X, y=None):
        X = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.freq_maps_ = {}
        for col in self.freq_cols:
            if col in X.columns:
                self.freq_maps_[col] = X[col].value_counts(normalize=True).to_dict()
        return self

    def transform(self, X):
        X = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        X = X.copy()

        # Time features
        if 'TransactionDT' in X.columns:
            X['tx_hour']    = (X['TransactionDT'] // 3600) % 24
            X['tx_weekday'] = (X['TransactionDT'] // (3600 * 24)) % 7
            X['tx_day']     = (X['TransactionDT'] // (3600 * 24)) % 30

        # Amount features
        if 'TransactionAmt' in X.columns:
            X['TransactionAmt_log']   = np.log1p(X['TransactionAmt'])
            X['TransactionAmt_cents'] = X['TransactionAmt'] - np.floor(X['TransactionAmt'])

        # Frequency encoding (using train-learned maps)
        for col in self.freq_cols:
            if col in X.columns and col in self.freq_maps_:
                X[f'{col}_freq'] = X[col].map(self.freq_maps_[col]).fillna(0)

        return X

In [ ]:
# Fit FE on train to inspect engineered columns before feature selection
fe_inspect = FraudFeatureEngineer(freq_cols=HIGH_CARD_COLS)
fe_inspect.fit(train_clean.drop(columns=['isFraud']))
train_fe = fe_inspect.transform(train_clean.drop(columns=['isFraud']))

new_features = (
    ['tx_hour', 'tx_weekday', 'tx_day', 'TransactionAmt_log', 'TransactionAmt_cents']
    + [f'{c}_freq' for c in HIGH_CARD_COLS]
)
new_features = [c for c in new_features if c in train_fe.columns]
print(f'New features added: {new_features}')
print(f'Total features after FE: {train_fe.shape[1]}')

In [ ]:
with mlflow.start_run(run_name='LR_Feature_Engineering'):
    mlflow.log_param('freq_encode_cols', str(HIGH_CARD_COLS))
    mlflow.log_param('new_features', str(new_features))
    mlflow.log_metric('n_new_features', len(new_features))
    mlflow.log_metric('n_features_after_fe', train_fe.shape[1])
    print('LR_Feature_Engineering run logged.')

# Feature Selection

In [ ]:
def calculate_iv(series, target, n_bins=10):
    """Information Value — measures predictive power of a feature."""
    total_events     = target.sum()
    total_non_events = len(target) - total_events
    if total_events == 0 or total_non_events == 0:
        return 0.0

    temp = pd.DataFrame({'f': series, 't': target}).dropna()
    if temp.empty:
        return 0.0

    if temp['f'].dtype == 'object' or temp['f'].nunique() < 20:
        groups = temp.groupby('f', observed=True)['t'].agg(['sum', 'count'])
    else:
        try:
            temp['bin'] = pd.qcut(temp['f'].rank(method='first'), n_bins, duplicates='drop')
            groups = temp.groupby('bin', observed=True)['t'].agg(['sum', 'count'])
        except Exception:
            return 0.0

    groups.columns = ['events', 'total']
    groups['non_events']     = groups['total'] - groups['events']
    groups['pct_events']     = groups['events'].clip(lower=1) / total_events
    groups['pct_non_events'] = groups['non_events'].clip(lower=1) / total_non_events
    groups['woe']            = np.log(groups['pct_events'] / groups['pct_non_events'])
    groups['iv']             = (groups['pct_events'] - groups['pct_non_events']) * groups['woe']
    return groups['iv'].sum()

In [ ]:
# Use a sample to speed up IV calculation on large dataset
y_train = train_clean['isFraud']

sample_idx = train_fe.sample(min(60000, len(train_fe)), random_state=RANDOM_STATE).index
X_sample = train_fe.loc[sample_idx]
y_sample = y_train.loc[sample_idx]

print('Calculating IV for all features...')
iv_scores = {}
for col in train_fe.columns:
    try:
        iv_scores[col] = calculate_iv(X_sample[col], y_sample)
    except Exception:
        iv_scores[col] = 0.0

iv_df = (pd.DataFrame.from_dict(iv_scores, orient='index', columns=['IV'])
           .sort_values('IV', ascending=False))

print('\nIV interpretation: <0.02 useless | 0.02-0.1 weak | 0.1-0.3 medium | 0.3-0.5 strong')
print(iv_df.head(30))

In [ ]:
IV_THRESHOLD = 0.02  # keep features with at least weak predictive power

selected_features = iv_df[iv_df['IV'] > IV_THRESHOLD].index.tolist()

NUMERIC_COLS = train_fe[selected_features].select_dtypes(include=[np.number]).columns.tolist()
CAT_COLS     = train_fe[selected_features].select_dtypes(include=['object']).columns.tolist()

print(f'Selected {len(selected_features)} features (IV > {IV_THRESHOLD})')
print(f'  Numeric:     {len(NUMERIC_COLS)}')
print(f'  Categorical: {len(CAT_COLS)}')

In [ ]:
with mlflow.start_run(run_name='LR_Feature_Selection'):
    mlflow.log_param('iv_threshold', IV_THRESHOLD)
    mlflow.log_param('selection_method', 'InformationValue')
    mlflow.log_metric('n_features_selected', len(selected_features))
    mlflow.log_metric('n_numeric_selected', len(NUMERIC_COLS))
    mlflow.log_metric('n_cat_selected', len(CAT_COLS))

    iv_df.to_csv('/tmp/lr_iv_scores.csv')
    mlflow.log_artifact('/tmp/lr_iv_scores.csv')
    print('LR_Feature_Selection run logged.')

# Training

In [ ]:
def build_pipeline(C, penalty='l2', solver='saga', max_iter=1000):
    """
    Returns a full sklearn Pipeline that works on raw (unprocessed) input data.
    Steps: FraudFeatureEngineer -> ColumnTransformer (impute+scale/encode) -> LogisticRegression
    """
    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ])
    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])
    preprocessor = ColumnTransformer([
        ('num', num_pipeline, NUMERIC_COLS),
        ('cat', cat_pipeline, CAT_COLS)
    ], remainder='drop')

    return Pipeline([
        ('fe',           FraudFeatureEngineer(freq_cols=HIGH_CARD_COLS)),
        ('preprocessor', preprocessor),
        ('clf',          LogisticRegression(
                            C=C,
                            penalty=penalty,
                            solver=solver,
                            max_iter=max_iter,
                            class_weight='balanced',
                            random_state=RANDOM_STATE,
                            n_jobs=-1
                         ))
    ])

In [ ]:
X_train = train_clean.drop(columns=['isFraud'])
y_train = train_clean['isFraud']

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
)

C = 0.1  # starting hyperparameter

with mlflow.start_run(run_name=f'LR_C{C}'):
    pipe = build_pipeline(C=C)
    pipe.fit(X_tr, y_tr)

    train_auc = roc_auc_score(y_tr,  pipe.predict_proba(X_tr)[:,  1])
    val_auc   = roc_auc_score(y_val, pipe.predict_proba(X_val)[:, 1])
    overfit_gap = train_auc - val_auc

    mlflow.log_param('C', C)
    mlflow.log_param('penalty', 'l2')
    mlflow.log_param('solver', 'saga')
    mlflow.log_param('max_iter', 1000)
    mlflow.log_param('class_weight', 'balanced')
    mlflow.log_param('n_features', len(selected_features))

    mlflow.log_metric('train_auc',   train_auc)
    mlflow.log_metric('val_auc',     val_auc)
    mlflow.log_metric('overfit_gap', overfit_gap)

    if overfit_gap > 0.05:
        fit_status = 'overfit'
    elif val_auc < 0.70:
        fit_status = 'underfit'
    else:
        fit_status = 'good_fit'
    mlflow.set_tag('fit_status', fit_status)

    print(f'train_auc={train_auc:.4f}  val_auc={val_auc:.4f}  gap={overfit_gap:.4f}  status={fit_status}')

    # Retrain on full data and register
    full_pipe = build_pipeline(C=C)
    full_pipe.fit(X_train, y_train)
    mlflow.sklearn.log_model(
        full_pipe,
        artifact_path='model',
        registered_model_name='LogisticRegression_FraudDetection'
    )
    best_run_id = mlflow.active_run().info.run_id

print(f'\nRun ID: {best_run_id}')

In [ ]:
print(f'Train AUC:    {train_auc:.4f}')
print(f'Val AUC:      {val_auc:.4f}')
print(f'Overfit gap:  {overfit_gap:.4f}')
print(f'Fit status:   {fit_status}')

In [ ]:
print(f'Best model: LogisticRegression_FraudDetection')
print(f'Best C={best_C}  |  Val AUC={best_val_auc:.4f}  |  Run ID={best_run_id}')